# Gold - Auditoria, Qualidade e IA

Esta camada reúne informações sobre o processo de classificação das justificativas de viagens.

O objetivo é permitir o acompanhamento da concordância entre regras e LLM, dos casos encaminhados para revisão humana e da decisão final utilizada no processo analítico.

In [0]:
df_llm = spark.table(
    "viagens_teste.silver.classificacao_llm_passagens"
)

df_revisao = spark.table(
    "viagens_teste.silver.revisao_humana_justificativas"
)

df_passagens = spark.table(
    "viagens_teste.silver.passagens"
)

print("CLASSIFICAÇÃO LLM")
df_llm.printSchema()

print("\nREVISÃO HUMANA")
df_revisao.printSchema()

print("\nPASSAGENS")
df_passagens.printSchema()

### Verificação das tabelas Silver disponíveis

Antes da construção da camada Gold de auditoria, verificamos se os resultados da classificação LLM em duas dimensões foram persistidos no catálogo.

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN viagens_teste.silver
    """)
)

### Leitura da classificação LLM em duas dimensões

Utilizamos a tabela persistida com as classificações de tipo de atividade e tema da justificativa para comparar as regras de negócio com o resultado do LLM.

In [0]:
df_llm_dupla = spark.table(
    "viagens_teste.silver.classificacao_llm_dupla_passagens"
)

df_llm_dupla.printSchema()

In [0]:
display(df_llm_dupla)

### Construção da base de auditoria

Integramos a classificação por regras, a classificação realizada pelo LLM, as revisões humanas e a decisão final persistida na Silver de passagens.

As concordâncias são calculadas dinamicamente, evitando indicadores fixos no dashboard.

In [0]:
from pyspark.sql import functions as F

df_passagens_base = (
    spark.table("viagens_teste.silver.passagens")
    .select(
        "id_solicitacao",
        "viajante",
        "data_viagem",
        "agencia",
        "centro_custo",
        "diretoria",
        "trimestre",
        "justificativa_original",
        "justificativa_limpa",
        "tipo_atividade_final",
        "tema_justificativa_final",
        "metodo_decisao_final"
    )
)

df_llm_dupla = (
    spark.table("viagens_teste.silver.classificacao_llm_dupla_passagens")
    .select(
        "id_solicitacao",
        "tipo_regra",
        "tipo_llm",
        "confianca_tipo_llm",
        "tema_regra",
        "tema_llm",
        "confianca_tema_llm"
    )
    .withColumn(
        "tipo_concorda",
        F.col("tipo_regra") == F.col("tipo_llm")
    )
    .withColumn(
        "tema_concorda",
        F.col("tema_regra") == F.col("tema_llm")
    )
)

df_revisao = (
    spark.table("viagens_teste.silver.revisao_humana_justificativas")
    .select(
        "id_solicitacao",
        "tipo_atividade_humano",
        "tema_justificativa_humano",
        "justificativa_revisao"
    )
)

### Integração das etapas de classificação

Cada solicitação passa a possuir a trilha completa de decisão: regra, LLM, eventual revisão humana e classificação final.

In [0]:
df_auditoria = (
    df_passagens_base
    .join(
        df_llm_dupla,
        on="id_solicitacao",
        how="left"
    )
    .join(
        df_revisao,
        on="id_solicitacao",
        how="left"
    )
    .withColumn(
        "revisao_humana",
        F.col("tipo_atividade_humano").isNotNull()
        | F.col("tema_justificativa_humano").isNotNull()
    )
    .withColumn(
        "divergencia_regra_llm",
        (~F.col("tipo_concorda"))
        | (~F.col("tema_concorda"))
    )
)

In [0]:
display(df_auditoria)

### Validação dos indicadores de auditoria

Validamos a quantidade de registros, concordâncias entre regras e LLM e os casos encaminhados para revisão humana.

In [0]:
total = df_auditoria.count()

tipo_concordantes = (
    df_auditoria
    .filter(F.col("tipo_concorda") == True)
    .count()
)

tema_concordantes = (
    df_auditoria
    .filter(F.col("tema_concorda") == True)
    .count()
)

revisoes_humanas = (
    df_auditoria
    .filter(F.col("revisao_humana") == True)
    .count()
)

print("Total de registros:", total)

print(
    "Concordância de Tipo:",
    tipo_concordantes,
    f"({tipo_concordantes / total * 100:.1f}%)"
)

print(
    "Concordância de Tema:",
    tema_concordantes,
    f"({tema_concordantes / total * 100:.1f}%)"
)

print(
    "Revisões Humanas:",
    revisoes_humanas
)

In [0]:
display(
    df_auditoria
    .filter(F.col("revisao_humana") == True)
    .select(
        "id_solicitacao",
        "tipo_regra",
        "tipo_llm",
        "tipo_atividade_humano",
        "tipo_atividade_final",
        "tema_regra",
        "tema_llm",
        "tema_justificativa_humano",
        "tema_justificativa_final",
        "justificativa_revisao"
    )
)

### Persistência da camada Gold de auditoria

Após a validação das classificações, concordâncias e revisões humanas, persistimos a tabela analítica utilizada pelo Power BI para monitoramento do processo de classificação.

In [0]:
(
    df_auditoria.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "viagens_teste.gold.auditoria_justificativas"
    )
)

In [0]:
df_gold_auditoria = spark.table(
    "viagens_teste.gold.auditoria_justificativas"
)

print(
    "Registros Gold:",
    df_gold_auditoria.count()
)

display(df_gold_auditoria)

In [0]:
print(
    "Sem classificação final de tipo:",
    df_gold_auditoria
    .filter(F.col("tipo_atividade_final").isNull())
    .count()
)

print(
    "Sem classificação final de tema:",
    df_gold_auditoria
    .filter(F.col("tema_justificativa_final").isNull())
    .count()
)